# Fine-tune Nuqta + CRNN-CTC on Kaggle GPU

This notebook trains both OCR candidates sequentially on the same official splits:

- **SAQR-handwriting**: `hw_image`;
- **SAQR-printed**: the paired `gt_image` with the same transcription;
- **KHATT-handwriting**: line-level KHATT from the Arabic Manuscript Collection.

Nuqta is adapted with LoRA. The original `FixFips/arabicocr-khatt` CRNN-CTC checkpoint is then fine-tuned with an expanded vocabulary. Validation and test results are reported separately for all three domains. Select an NVIDIA GPU and enable Internet.


In [ ]:
%pip install -q -U "transformers==5.8.1" "peft>=0.20.0" "datasets>=4.4.0" accelerate huggingface_hub hf_xet safetensors pandas pillow numpy "arabicocr-khatt==0.1.0" rapidfuzz


## Experiment controls

Start with `FAST_DEV_RUN = True`. The default full run performs the Nuqta LR sweep, two Nuqta epochs, then up to twenty CRNN epochs with early stopping.

In [ ]:
USE_SAQR = True
USE_KHATT = True
RUN_LR_SWEEP = True
LEARNING_RATE = 2e-5
LR_CANDIDATES = [5e-6, 2e-5, 8e-5]
PILOT_STEPS = 120
PILOT_TRAIN_SAMPLES = 1600
PILOT_EVAL_PER_SOURCE = 48
NUQTA_EPOCHS = 2.0
FINAL_EVAL_PER_SOURCE = 256
RUN_CRNN = True
CRNN_EPOCHS = 20
CRNN_PATIENCE = 5
CRNN_LEARNING_RATE = 3e-4
CRNN_BATCH_SIZE = 8
FAST_DEV_RUN = True


## Embedded GPU implementation

The repository keeps the readable source in `kaggle/train_ocr_gpu.py`; this embedded copy makes the notebook portable when uploaded alone.

In [ ]:
%%writefile /kaggle/working/train_ocr_gpu.py
"""Fine-tune Nuqta and ArabicOCR CRNN-CTC on a Kaggle NVIDIA GPU."""

from __future__ import annotations

import argparse
import gc
import json
import math
import os
import re
import shutil
import time
import unicodedata
import zipfile
from collections import defaultdict
from pathlib import Path
from typing import Any

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from arabicocr_khatt.augment import ArabicAugment
from arabicocr_khatt.metrics import norm_ar
from arabicocr_khatt.model import CRNN, build_bigram_lm, ctc_greedy_decode, text_to_ids
from arabicocr_khatt.pipeline import load_checkpoint, save_bigram_lm_json
from arabicocr_khatt.preprocess import (
    binarize,
    normalize,
    pad_width,
    resize_keep_ratio_height,
    to_grayscale,
)
from datasets import Dataset, Image as HFImage, concatenate_datasets, load_dataset
from huggingface_hub import hf_hub_download, snapshot_download
from peft import LoraConfig, get_peft_model
from PIL import Image, ImageOps
from torch.utils.data import DataLoader, Dataset as TorchDataset
from transformers import (
    AutoModelForMultimodalLM,
    AutoProcessor,
    Trainer,
    TrainingArguments,
    set_seed,
)


MODEL_ID = "SalahALHaismawi/Nuqta-Arabic-OCR-0.8B"
KHATT_REPO = "TheSeniorTeam/Arabic_Manuscript_Collection_Dataset"
PROMPT = (
    "Transcribe the Arabic text in this image exactly as written. Include only "
    "the diacritics (tashkeel) that are visible in the image."
)
SEED = 42

# Fixed shapes cap VRAM usage and make the GPU trials directly comparable to
# the TPU notebook.
IMAGE_WIDTH = 2048
IMAGE_HEIGHT = 256
TRAIN_MAX_LENGTH = 1024
GENERATION_TOKENS = 256

ARABIC_MARKS = re.compile(r"[\u064B-\u065F\u0670]")


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--work-dir", default="/kaggle/working/ocr_mixed")
    parser.add_argument("--output-dir", default="/kaggle/working/arabic-ocr-finetuned")
    parser.add_argument("--no-saqr", action="store_true")
    parser.add_argument("--no-khatt", action="store_true")
    parser.add_argument("--saqr-repeat", type=int, default=2)
    parser.add_argument("--skip-lr-sweep", action="store_true")
    parser.add_argument("--learning-rate", type=float, default=2e-5)
    parser.add_argument("--lr-candidates", nargs="+", type=float, default=[5e-6, 2e-5, 8e-5])
    parser.add_argument("--pilot-steps", type=int, default=120)
    parser.add_argument("--pilot-train-samples", type=int, default=1600)
    parser.add_argument("--pilot-eval-per-source", type=int, default=48)
    parser.add_argument("--epochs", type=float, default=2.0)
    parser.add_argument("--final-eval-per-source", type=int, default=256)
    parser.add_argument("--skip-crnn", action="store_true")
    parser.add_argument("--crnn-epochs", type=int, default=20)
    parser.add_argument("--crnn-patience", type=int, default=5)
    parser.add_argument("--crnn-learning-rate", type=float, default=3e-4)
    parser.add_argument("--crnn-batch-size", type=int, default=8)
    parser.add_argument("--fast-dev-run", action="store_true")
    return parser.parse_args()


def normalize_config(args: argparse.Namespace) -> dict[str, Any]:
    if args.no_saqr and args.no_khatt:
        raise ValueError("At least one of SAQR or KHATT must be enabled.")
    config = vars(args).copy()
    config["work_dir"] = str(Path(args.work_dir))
    config["output_dir"] = str(Path(args.output_dir))
    config["use_saqr"] = not args.no_saqr
    config["use_khatt"] = not args.no_khatt
    config["run_lr_sweep"] = not args.skip_lr_sweep
    if args.fast_dev_run:
        config.update(
            pilot_steps=3,
            pilot_train_samples=24,
            pilot_eval_per_source=4,
            epochs=0.01,
            final_eval_per_source=4,
            crnn_epochs=1,
        )
    return config


def keep_columns(dataset: Dataset, image_column: str, source: str) -> Dataset:
    dataset = dataset.select_columns([image_column, "text"])
    if image_column != "image":
        dataset = dataset.rename_column(image_column, "image")
    return dataset.map(lambda _: {"source": source}, desc=f"Tag {source}")


def load_saqr() -> dict[str, Dataset]:
    raw = load_dataset("mahmoudsalah01/SAQR")
    # SAQR is paired: every transcription has a handwritten image and its
    # printed ground-truth rendering. Keeping the official split before
    # concatenating the two views prevents pair leakage across splits.
    result = {}
    for split in ("train", "validation", "test"):
        result[split] = concatenate_datasets(
            [
                keep_columns(raw[split], "hw_image", "SAQR-handwriting"),
                keep_columns(raw[split], "gt_image", "SAQR-printed"),
            ]
        )
    return result


def load_khatt(work_dir: Path) -> dict[str, Dataset]:
    root = work_dir / "KHATT"
    root.mkdir(parents=True, exist_ok=True)
    archive = hf_hub_download(KHATT_REPO, "KHATT/images.zip", repo_type="dataset")
    marker = root / ".extracted"
    if not marker.exists():
        with zipfile.ZipFile(archive) as handle:
            handle.extractall(root)
        marker.touch()

    result: dict[str, Dataset] = {}
    fallback_index: dict[str, Path] | None = None
    for split in ("train", "val", "test"):
        csv_path = hf_hub_download(
            KHATT_REPO, f"KHATT/{split}.csv", repo_type="dataset"
        )
        frame = pd.read_csv(csv_path).dropna(subset=["image_path", "text"])
        paths = [root / "images" / split / name for name in frame["image_path"]]
        if any(not path.is_file() for path in paths):
            if fallback_index is None:
                fallback_index = {
                    path.name: path for path in root.rglob("*") if path.is_file()
                }
            paths = [fallback_index.get(Path(name).name) for name in frame["image_path"]]
        missing = [
            name
            for name, path in zip(frame["image_path"], paths)
            if path is None or not path.is_file()
        ]
        if missing:
            raise FileNotFoundError(f"KHATT images missing, first examples: {missing[:5]}")
        dataset = Dataset.from_dict(
            {
                "image": [str(path) for path in paths],
                "text": frame["text"].astype(str).tolist(),
                "source": ["KHATT-handwriting"] * len(frame),
            }
        ).cast_column("image", HFImage())
        result["validation" if split == "val" else split] = dataset
    return result


def limited(dataset: Dataset, limit: int | None) -> Dataset:
    if limit is None or len(dataset) <= limit:
        return dataset
    return dataset.shuffle(seed=SEED).select(range(limit))


def per_source_subset(dataset: Dataset, source: str, limit: int | None) -> Dataset:
    indices = [index for index, value in enumerate(dataset["source"]) if value == source]
    return limited(dataset.select(indices), limit)


def prepare_datasets(config: dict[str, Any]) -> dict[str, Dataset]:
    work_dir = Path(config["work_dir"])
    sources = []
    if config["use_saqr"]:
        sources.append(load_saqr())
    if config["use_khatt"]:
        sources.append(load_khatt(work_dir))

    train_parts: list[Dataset] = []
    validation_parts: list[Dataset] = []
    test_parts: list[Dataset] = []
    for source in sources:
        repeat = (
            config["saqr_repeat"]
            if str(source["train"]["source"][0]).startswith("SAQR-")
            else 1
        )
        train_parts.extend([source["train"]] * repeat)
        validation_parts.append(source["validation"])
        test_parts.append(source["test"])

    train_data = concatenate_datasets(train_parts).shuffle(seed=SEED)
    validation_data = concatenate_datasets(validation_parts)
    test_data = concatenate_datasets(test_parts)
    pilot_parts = [
        per_source_subset(validation_data, source, config["pilot_eval_per_source"])
        for source in sorted(set(validation_data["source"]))
    ]
    pilot_validation = concatenate_datasets(pilot_parts)
    final_validation_parts = [
        per_source_subset(validation_data, source, config["final_eval_per_source"])
        for source in sorted(set(validation_data["source"]))
    ]
    final_test_parts = [
        per_source_subset(test_data, source, config["final_eval_per_source"])
        for source in sorted(set(test_data["source"]))
    ]
    bundle = {
        "train": train_data,
        "validation": validation_data,
        "test": test_data,
        "pilot_train": limited(train_data, config["pilot_train_samples"]),
        "pilot_validation": pilot_validation,
        "final_validation": concatenate_datasets(final_validation_parts),
        "final_test": concatenate_datasets(final_test_parts),
    }
    if config["fast_dev_run"]:
        bundle["train"] = limited(train_data, 64)
        bundle["pilot_train"] = limited(bundle["train"], 24)
        bundle["validation"] = bundle["final_validation"]
        bundle["test"] = bundle["final_test"]
    print({name: len(dataset) for name, dataset in bundle.items()})
    print(pd.Series(train_data["source"]).value_counts())
    return bundle


def prepare_line_image(image: Image.Image) -> Image.Image:
    """Crop whitespace, preserve aspect ratio, and pad to a stable shape."""
    gray = ImageOps.autocontrast(image.convert("L"))
    # A conservative mask removes only near-white outer margins. The extra
    # border protects ascenders, descenders, dots and visible tashkeel.
    ink_mask = gray.point(lambda pixel: 255 if pixel < 248 else 0)
    box = ink_mask.getbbox()
    if box:
        left, top, right, bottom = box
        border = 8
        gray = gray.crop(
            (
                max(0, left - border),
                max(0, top - border),
                min(gray.width, right + border),
                min(gray.height, bottom + border),
            )
        )
    usable_width = IMAGE_WIDTH - 32
    usable_height = IMAGE_HEIGHT - 32
    scale = min(usable_width / max(1, gray.width), usable_height / max(1, gray.height))
    new_size = (
        max(1, round(gray.width * scale)),
        max(1, round(gray.height * scale)),
    )
    resized = gray.resize(new_size, Image.Resampling.LANCZOS)
    canvas = Image.new("L", (IMAGE_WIDTH, IMAGE_HEIGHT), 255)
    # Right alignment matches Arabic reading direction without changing pixels.
    x = IMAGE_WIDTH - 16 - resized.width
    y = (IMAGE_HEIGHT - resized.height) // 2
    canvas.paste(resized, (x, y))
    return canvas.convert("RGB")


def user_messages() -> list[dict[str, Any]]:
    return [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": PROMPT},
            ],
        }
    ]


def full_messages(target: str) -> list[dict[str, Any]]:
    return user_messages() + [
        {"role": "assistant", "content": [{"type": "text", "text": str(target)}]}
    ]


def create_processor(model_path: str):
    processor = AutoProcessor.from_pretrained(model_path, local_files_only=True)
    processor.tokenizer.padding_side = "right"
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token
    return processor


class OCRCollator:
    def __init__(self, processor) -> None:
        self.processor = processor

    def __call__(self, examples: list[dict[str, Any]]) -> dict[str, torch.Tensor]:
        images = [prepare_line_image(example["image"]) for example in examples]
        full_texts = [
            self.processor.apply_chat_template(
                full_messages(example["text"]),
                tokenize=False,
                add_generation_prompt=False,
            )
            for example in examples
        ]
        prompt_texts = [
            self.processor.apply_chat_template(
                user_messages(), tokenize=False, add_generation_prompt=True
            )
            for _ in examples
        ]
        common = {
            "images": images,
            "padding": "max_length",
            "truncation": True,
            "max_length": TRAIN_MAX_LENGTH,
            "return_tensors": "pt",
        }
        batch = self.processor(text=full_texts, **common)
        prompt_batch = self.processor(text=prompt_texts, **common)
        labels = batch["input_ids"].clone()
        labels[batch["attention_mask"] == 0] = -100
        for row, prompt_length in enumerate(prompt_batch["attention_mask"].sum(dim=1).tolist()):
            labels[row, : int(prompt_length)] = -100
        if any(int((row != -100).sum()) == 0 for row in labels):
            raise ValueError(
                "A target was fully truncated. Increase TRAIN_MAX_LENGTH or reduce image size."
            )
        batch["labels"] = labels
        return batch


def build_model(model_path: str):
    major = torch.cuda.get_device_capability()[0]
    compute_dtype = torch.bfloat16 if major >= 8 else torch.float16
    model = AutoModelForMultimodalLM.from_pretrained(
        model_path,
        dtype=compute_dtype,
        attn_implementation="sdpa",
        low_cpu_mem_usage=True,
        local_files_only=True,
    )
    model.config.use_cache = False
    model.enable_input_require_grads()
    lora = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        target_modules="all-linear",
        task_type="CAUSAL_LM",
        use_rslora=True,
    )
    return get_peft_model(model, lora)


def normalize_metric(text: str, strip_marks: bool) -> str:
    text = unicodedata.normalize("NFKC", str(text)).replace("ـ", "")
    if strip_marks:
        text = ARABIC_MARKS.sub("", text)
    text = re.sub(r"[^\w\u0600-\u06FF]+", " ", text, flags=re.UNICODE)
    return re.sub(r"\s+", " ", text).strip()


def edit_distance(reference: list[Any] | str, hypothesis: list[Any] | str) -> int:
    previous = list(range(len(hypothesis) + 1))
    for i, ref_item in enumerate(reference, 1):
        current = [i]
        for j, hyp_item in enumerate(hypothesis, 1):
            current.append(
                min(
                    current[-1] + 1,
                    previous[j] + 1,
                    previous[j - 1] + (ref_item != hyp_item),
                )
            )
        previous = current
    return previous[-1]


def corpus_error_rates(
    predictions: list[str], references: list[str], strip_marks: bool
) -> dict[str, float]:
    char_edits = char_total = word_edits = word_total = 0
    for prediction, reference in zip(predictions, references):
        prediction = normalize_metric(prediction, strip_marks)
        reference = normalize_metric(reference, strip_marks)
        pred_chars, ref_chars = prediction.replace(" ", ""), reference.replace(" ", "")
        pred_words, ref_words = prediction.split(), reference.split()
        char_edits += edit_distance(ref_chars, pred_chars)
        char_total += len(ref_chars)
        word_edits += edit_distance(ref_words, pred_words)
        word_total += len(ref_words)
    return {
        "cer": char_edits / max(1, char_total),
        "wer": word_edits / max(1, word_total),
    }


@torch.inference_mode()
def predict_one(model, processor, image: Image.Image) -> str:
    prompt = processor.apply_chat_template(
        user_messages(), tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        images=[prepare_line_image(image)], text=[prompt], return_tensors="pt"
    )
    inputs = inputs.to(next(model.parameters()).device)
    output = model.generate(
        **inputs,
        do_sample=False,
        max_new_tokens=GENERATION_TOKENS,
        repetition_penalty=1.02,
    )
    generated = output[:, inputs["input_ids"].shape[1] :]
    text = processor.batch_decode(generated, skip_special_tokens=True)[0]
    return re.sub(
        r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE
    ).strip()


def score_rows(rows: list[dict[str, str]]) -> dict[str, Any]:
    predictions = [row["prediction"] for row in rows]
    references = [row["reference"] for row in rows]
    marked = corpus_error_rates(predictions, references, strip_marks=False)
    plain = corpus_error_rates(predictions, references, strip_marks=True)
    return {
        "n": len(rows),
        "cer_with_marks": marked["cer"],
        "wer_with_marks": marked["wer"],
        "cer_no_marks": plain["cer"],
        "wer_no_marks": plain["wer"],
        "examples": [
            [row["reference"], row["prediction"]] for row in rows[:5]
        ],
    }


def summarize_rows(rows: list[dict[str, str]]) -> dict[str, Any]:
    grouped: dict[str, list[dict[str, str]]] = defaultdict(list)
    for row in rows:
        grouped[row["source"]].append(row)
    return {
        "overall": score_rows(rows),
        "by_source": {source: score_rows(items) for source, items in sorted(grouped.items())},
    }


def evaluate_nuqta(model, processor, dataset: Dataset, label: str) -> dict[str, Any]:
    model.eval()
    model.config.use_cache = True
    rows = []
    for index, example in enumerate(dataset, 1):
        rows.append({
            "source": str(example["source"]),
            "reference": str(example["text"]),
            "prediction": predict_one(model, processor, example["image"]),
        })
        if index % 25 == 0:
            print(f"{label}: {index}/{len(dataset)}")
    model.config.use_cache = False
    return summarize_rows(rows)


def training_args(
    output_dir: Path,
    learning_rate: float,
    *,
    max_steps: int = -1,
    epochs: float = 1.0,
    save: bool = False,
) -> TrainingArguments:
    use_bf16 = torch.cuda.get_device_capability()[0] >= 8
    return TrainingArguments(
        output_dir=str(output_dir),
        overwrite_output_dir=True,
        max_steps=max_steps,
        num_train_epochs=epochs,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=learning_rate,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        weight_decay=0.01,
        logging_steps=10,
        save_strategy="steps" if save else "no",
        save_steps=250,
        save_total_limit=2,
        fp16=not use_bf16,
        bf16=use_bf16,
        tf32=use_bf16,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        remove_unused_columns=False,
        report_to="none",
        dataloader_num_workers=0,
        optim="adamw_torch",
        seed=SEED,
    )


def train_nuqta_once(
    config: dict[str, Any],
    datasets: dict[str, Dataset],
    model_path: str,
    learning_rate: float,
    *,
    trial_name: str,
    full: bool,
) -> tuple[Trainer, Any, Any, Any, float]:
    set_seed(SEED)
    processor = create_processor(model_path)
    collator = OCRCollator(processor)
    model = build_model(model_path)
    output_dir = (
        Path(config["output_dir"]) / "nuqta-lora"
        if full
        else Path(config["work_dir"]) / trial_name
    )
    trainer = Trainer(
        model=model,
        args=training_args(
            output_dir,
            learning_rate,
            max_steps=-1 if full else config["pilot_steps"],
            epochs=config["epochs"] if full else 1.0,
            save=full,
        ),
        train_dataset=datasets["train"] if full else datasets["pilot_train"],
        data_collator=collator,
    )
    result = trainer.train()
    return trainer, model, processor, collator, float(result.training_loss)


def run_nuqta_gpu(
    config: dict[str, Any], datasets: dict[str, Dataset], model_path: str
) -> dict[str, Any]:
    output_dir = Path(config["output_dir"]) / "nuqta-lora"
    sweep_results = []
    if config["run_lr_sweep"]:
        for learning_rate in config["lr_candidates"]:
            tag = f"pilot-{learning_rate:g}".replace(".", "p")
            trainer, model, processor, collator, train_loss = train_nuqta_once(
                config, datasets, model_path, learning_rate,
                trial_name=tag, full=False,
            )
            metrics = evaluate_nuqta(
                model, processor, datasets["pilot_validation"], tag
            )
            sweep_results.append({
                "learning_rate": learning_rate,
                "train_loss": train_loss,
                **metrics,
            })
            shutil.rmtree(Path(config["work_dir"]) / tag, ignore_errors=True)
            del trainer, model, processor, collator
            gc.collect()
            torch.cuda.empty_cache()

    selected_lr = (
        float(min(sweep_results, key=lambda row: row["overall"]["cer_no_marks"])["learning_rate"])
        if sweep_results else float(config["learning_rate"])
    )
    print("Selected Nuqta learning rate:", selected_lr)
    trainer, model, processor, collator, full_train_loss = train_nuqta_once(
        config, datasets, model_path, selected_lr, trial_name="full", full=True
    )
    trainer.save_model(output_dir)
    processor.save_pretrained(output_dir)
    validation_metrics = evaluate_nuqta(
        model, processor, datasets["final_validation"], "Nuqta validation"
    )
    test_metrics = evaluate_nuqta(
        model, processor, datasets["final_test"], "Nuqta test"
    )
    report = {
        "base_model": MODEL_ID,
        "runtime": torch.cuda.get_device_name(0),
        "domains": ["SAQR-handwriting", "SAQR-printed", "KHATT-handwriting"],
        "selected_lr": selected_lr,
        "lr_sweep": sweep_results,
        "full_train_loss": full_train_loss,
        "metrics": {"validation": validation_metrics, "test": test_metrics},
    }
    with (output_dir / "experiment_report.json").open("w", encoding="utf-8") as handle:
        json.dump(report, handle, ensure_ascii=False, indent=2)
    del trainer, model, processor, collator
    gc.collect()
    torch.cuda.empty_cache()
    return report


CRNN_HEIGHT = 96
CRNN_MAX_WIDTH = 1536
CRNN_GRAD_ACCUM = 4


class MixedCRNNDataset(TorchDataset):
    def __init__(self, dataset: Dataset, augment=None) -> None:
        self.dataset = dataset
        self.augment = augment

    def __len__(self) -> int:
        return len(self.dataset)

    def __getitem__(self, index: int):
        example = self.dataset[index]
        image = normalize(binarize(to_grayscale(example["image"])))
        if self.augment is not None:
            image = self.augment(image)
        image = resize_keep_ratio_height(image, CRNN_HEIGHT)
        if image.width > CRNN_MAX_WIDTH:
            image = image.resize(
                (CRNN_MAX_WIDTH, CRNN_HEIGHT), Image.Resampling.LANCZOS
            )
        image = pad_width(image, CRNN_MAX_WIDTH)
        array = np.asarray(image, dtype=np.float32) / 255.0
        return (
            torch.from_numpy(array).unsqueeze(0),
            str(example["text"]),
            str(example["source"]),
            index,
        )


class CRNNCollator:
    def __init__(self, char2id: dict[str, int], unk_id: int) -> None:
        self.char2id = char2id
        self.unk_id = unk_id

    def __call__(self, batch):
        images, texts_rtl, sources, indices = zip(*batch)
        encoded = [
            torch.tensor(
                text_to_ids(text[::-1], self.char2id, self.unk_id),
                dtype=torch.long,
            )
            for text in texts_rtl
        ]
        lengths = torch.tensor([len(item) for item in encoded], dtype=torch.long)
        return (
            torch.stack(images), torch.cat(encoded), lengths,
            list(texts_rtl), list(sources), list(indices),
        )


def add_normalized_wer(metrics: dict[str, Any], rows: list[dict[str, str]]) -> None:
    groups = {"overall": rows}
    groups.update({
        source: [row for row in rows if row["source"] == source]
        for source in metrics["by_source"]
    })
    for name, selected in groups.items():
        values = metrics["overall"] if name == "overall" else metrics["by_source"][name]
        references = [norm_ar(row["reference"]) for row in selected]
        predictions = [norm_ar(row["prediction"]) for row in selected]
        values["wer_normalized"] = corpus_error_rates(
            predictions, references, strip_marks=False
        )["wer"]


@torch.inference_mode()
def evaluate_crnn(model, loader, id2char: dict[int, str]):
    model.eval()
    rows = []
    for images, _, _, texts, sources, indices in loader:
        logits = model(images.cuda(non_blocking=True))
        predictions = [text[::-1] for text in ctc_greedy_decode(logits, id2char)]
        rows.extend(
            {
                "id": f"{source}:{index}",
                "source": source,
                "reference": reference,
                "prediction": prediction,
            }
            for source, index, reference, prediction in zip(
                sources, indices, texts, predictions
            )
        )
    metrics = summarize_rows(rows)
    add_normalized_wer(metrics, rows)
    return metrics, rows


def load_crnn_for_finetuning(checkpoint_path: Path, datasets: dict[str, Dataset]):
    state = load_checkpoint(checkpoint_path)
    base_vocab = list(state["vocab"])
    dataset_chars = sorted(set(
        "".join(datasets["train"]["text"])
        + "".join(datasets["validation"]["text"])
    ))
    vocab = base_vocab + [char for char in dataset_chars if char not in base_vocab]
    char2id = {char: index for index, char in enumerate(vocab)}
    id2char = {index: char for index, char in enumerate(vocab)}
    model = CRNN(num_classes=len(vocab), use_attention=False)
    fresh = model.state_dict()
    for key, value in state["model"].items():
        if key in fresh and fresh[key].shape == value.shape:
            fresh[key] = value
    model.load_state_dict(fresh)
    if len(vocab) != len(base_vocab):
        with torch.no_grad():
            model.fc.weight[:len(base_vocab)].copy_(state["model"]["fc.weight"])
            model.fc.bias[:len(base_vocab)].copy_(state["model"]["fc.bias"])
    return (
        model.cuda(), vocab, char2id, id2char, char2id.get("<unk>", 1)
    )


def make_crnn_loader(dataset, collator, batch_size, *, train: bool):
    return DataLoader(
        MixedCRNNDataset(dataset, ArabicAugment(training=True) if train else None),
        batch_size=batch_size,
        shuffle=train,
        num_workers=2,
        pin_memory=True,
        persistent_workers=True,
        collate_fn=collator,
    )


def run_crnn_gpu(
    config: dict[str, Any], datasets: dict[str, Dataset], checkpoint_path: Path
) -> dict[str, Any]:
    output_dir = Path(config["output_dir"]) / "crnn-ctc"
    output_dir.mkdir(parents=True, exist_ok=True)
    model, vocab, char2id, id2char, unk_id = load_crnn_for_finetuning(
        checkpoint_path, datasets
    )
    collator = CRNNCollator(char2id, unk_id)
    train_loader = make_crnn_loader(
        datasets["train"], collator, config["crnn_batch_size"], train=True
    )
    validation_loader = make_crnn_loader(
        datasets["validation"], collator, config["crnn_batch_size"], train=False
    )
    test_loader = make_crnn_loader(
        datasets["test"], collator, config["crnn_batch_size"], train=False
    )
    criterion = nn.CTCLoss(blank=0, zero_infinity=True)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=config["crnn_learning_rate"] / 10, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=config["crnn_learning_rate"],
        epochs=config["crnn_epochs"],
        steps_per_epoch=math.ceil(len(train_loader) / CRNN_GRAD_ACCUM),
        pct_start=0.1,
        anneal_strategy="cos",
        div_factor=10,
        final_div_factor=100,
    )
    use_bf16 = torch.cuda.get_device_capability()[0] >= 8
    scaler = torch.cuda.amp.GradScaler(enabled=not use_bf16)
    amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
    best_checkpoint = output_dir / "crnn_best.pt"
    history = []
    best_cer = float("inf")
    stale = 0

    for epoch in range(1, config["crnn_epochs"] + 1):
        started = time.perf_counter()
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0
        for step, (images, targets, target_lengths, _, _, _) in enumerate(train_loader, 1):
            images = images.cuda(non_blocking=True)
            targets = targets.cuda(non_blocking=True)
            target_lengths = target_lengths.cuda(non_blocking=True)
            with torch.autocast("cuda", dtype=amp_dtype):
                logits = model(images)
                timesteps, batch_size, _ = logits.shape
                input_lengths = torch.full(
                    (batch_size,), timesteps, dtype=torch.long, device="cuda"
                )
                loss = criterion(
                    logits.float().log_softmax(2), targets,
                    input_lengths, target_lengths,
                )
            scaler.scale(loss / CRNN_GRAD_ACCUM).backward()
            running_loss += float(loss.detach())
            if step % CRNN_GRAD_ACCUM == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

        validation_metrics, validation_rows = evaluate_crnn(
            model, validation_loader, id2char
        )
        row = {
            "epoch": epoch,
            "train_loss": running_loss / max(1, len(train_loader)),
            **{
                key: value for key, value in validation_metrics["overall"].items()
                if key != "examples"
            },
            "seconds": time.perf_counter() - started,
        }
        history.append(row)
        pd.DataFrame(history).to_csv(output_dir / "metrics.csv", index=False)
        print(
            f"CRNN epoch {epoch:02d}: loss={row['train_loss']:.4f}, "
            f"CER(no marks)={row['cer_no_marks']:.4f}"
        )
        if row["cer_no_marks"] < best_cer:
            best_cer = row["cer_no_marks"]
            stale = 0
            torch.save({
                "model": model.state_dict(), "vocab": vocab,
                "arch_version": 2, "use_attention": False,
            }, best_checkpoint)
            pd.DataFrame(validation_rows).to_csv(
                output_dir / "validation_predictions.csv", index=False
            )
        else:
            stale += 1
            if stale >= config["crnn_patience"]:
                print("CRNN early stopping")
                break

    best_state = load_checkpoint(best_checkpoint)
    model.load_state_dict(best_state["model"])
    validation_metrics, _ = evaluate_crnn(model, validation_loader, id2char)
    test_metrics, test_rows = evaluate_crnn(model, test_loader, id2char)
    pd.DataFrame(test_rows).to_csv(output_dir / "test_predictions.csv", index=False)
    bigram_lm = build_bigram_lm(
        [text[::-1] for text in datasets["train"]["text"]], char2id
    )
    save_bigram_lm_json(bigram_lm, output_dir / "bigram_lm.json")
    report = {
        "base_model": "FixFips/arabicocr-khatt",
        "selection_metric": "validation cer_no_marks",
        "best_cer_no_marks": best_cer,
        "history": history,
        "metrics": {"validation": validation_metrics, "test": test_metrics},
    }
    with (output_dir / "experiment_report.json").open("w", encoding="utf-8") as handle:
        json.dump(report, handle, ensure_ascii=False, indent=2)
    return report


def main() -> None:
    config = normalize_config(parse_args())
    set_seed(SEED)
    np.random.seed(SEED)
    work_dir = Path(config["work_dir"])
    output_dir = Path(config["output_dir"])
    work_dir.mkdir(parents=True, exist_ok=True)
    shutil.rmtree(output_dir, ignore_errors=True)
    output_dir.mkdir(parents=True, exist_ok=True)
    if not torch.cuda.is_available():
        raise RuntimeError("Select a Kaggle NVIDIA GPU accelerator.")
    print("GPU:", torch.cuda.get_device_name(0))

    model_path = snapshot_download(MODEL_ID)
    crnn_checkpoint = Path(
        hf_hub_download("FixFips/arabicocr-khatt", "crnn_best.pt")
    )
    datasets = prepare_datasets(config)
    preflight_processor = create_processor(model_path)
    preflight = OCRCollator(preflight_processor)([datasets["train"][0]])
    print("Preflight:", {
        key: tuple(value.shape) for key, value in preflight.items()
        if hasattr(value, "shape")
    })
    del preflight, preflight_processor
    gc.collect()

    nuqta_report = run_nuqta_gpu(config, datasets, model_path)
    crnn_report = None
    if not config["skip_crnn"]:
        crnn_report = run_crnn_gpu(config, datasets, crnn_checkpoint)
    combined = {
        "datasets": ["SAQR-handwriting", "SAQR-printed", "KHATT-handwriting"],
        "nuqta": nuqta_report,
        "crnn_ctc": crnn_report,
    }
    with (output_dir / "experiment_report.json").open("w", encoding="utf-8") as handle:
        json.dump(combined, handle, ensure_ascii=False, indent=2)
    archive = shutil.make_archive(str(output_dir), "zip", root_dir=output_dir)
    print("Combined output:", output_dir)
    print("Download:", archive)


if __name__ == "__main__":
    main()



In [ ]:
import subprocess
import sys

command = [
    sys.executable, "/kaggle/working/train_ocr_gpu.py",
    "--learning-rate", str(LEARNING_RATE),
    "--lr-candidates", *map(str, LR_CANDIDATES),
    "--pilot-steps", str(PILOT_STEPS),
    "--pilot-train-samples", str(PILOT_TRAIN_SAMPLES),
    "--pilot-eval-per-source", str(PILOT_EVAL_PER_SOURCE),
    "--epochs", str(NUQTA_EPOCHS),
    "--final-eval-per-source", str(FINAL_EVAL_PER_SOURCE),
    "--crnn-epochs", str(CRNN_EPOCHS),
    "--crnn-patience", str(CRNN_PATIENCE),
    "--crnn-learning-rate", str(CRNN_LEARNING_RATE),
    "--crnn-batch-size", str(CRNN_BATCH_SIZE),
]
if not USE_SAQR:
    command.append("--no-saqr")
if not USE_KHATT:
    command.append("--no-khatt")
if not RUN_LR_SWEEP:
    command.append("--skip-lr-sweep")
if not RUN_CRNN:
    command.append("--skip-crnn")
if FAST_DEV_RUN:
    command.append("--fast-dev-run")
print("Running:", " ".join(command))
subprocess.run(command, check=True)


In [ ]:
import json
from pathlib import Path
import pandas as pd

output_dir = Path("/kaggle/working/arabic-ocr-finetuned")
report = json.loads((output_dir / "experiment_report.json").read_text(encoding="utf-8"))
print("Domains:", report["datasets"])
for model_name in ("nuqta", "crnn_ctc"):
    model_report = report.get(model_name)
    if not model_report:
        continue
    print(f"\n## {model_name}")
    for split, metrics in model_report["metrics"].items():
        print(split)
        display(pd.DataFrame(metrics["by_source"]).T)
print("Output:", output_dir)
print("ZIP:", output_dir.with_suffix(".zip"))


## Outputs

- `nuqta-lora/`: LoRA adapter, processor, and Nuqta report.
- `crnn-ctc/`: `crnn_best.pt`, bigram LM, predictions, epoch metrics, and CRNN report.
- `experiment_report.json`: combined comparison.
- `/kaggle/working/arabic-ocr-finetuned.zip`: downloadable bundle.
